# 🧹 NLP Preprocessing (PT-BR): Parallel Cleaning
Este notebook realiza a limpeza e normalização de comentários do YouTube em paralelo, focando em regex e remoção de ruído para performance.

In [ ]:
import pandas as pd
import re
from unidecode import unidecode
from pandarallel import pandarallel

# Inicializar processamento paralelo
pandarallel.initialize(progress_bar=False)

# 1. Carregar dados
df = pd.read_parquet('bundesliga_2425_cazetv_chat.parquet')
print(f"Dataset carregado: {len(df):,} comentários.")

## 2. Pipeline de Limpeza e Normalização

In [ ]:
pt_br_slang = {
    "vc": "voce", "vcs": "voces", "pq": "porque", "ta": "esta",
    "tava": "estava", "mt": "muito", "obg": "obrigado", "p": "para", "q": "que"
}

def clean_message(text):
    if not isinstance(text, str): return None
    
    # 1. Lowercasing
    text = text.lower()
    
    # 2. Remover URLs e Menções
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@[\w:]+', '', text)
    
    # 3. Remover caracteres especiais (ASCII)
    text = unidecode(text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 4. Normalizar palavras intensificadas (gooooooal -> goall)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # 5. Expansão de Gírias
    words = text.split()
    words = [pt_br_slang.get(w, w) for w in words]
    
    # 6. Filtragem de Token Count (>5 tokens)
    if len(words) < 5: return None
    
    return " ".join(words)

print("Iniciando processamento paralelo...")
df['mensagem_limpa'] = df['mensagem'].parallel_apply(clean_message)

df_final = df.dropna(subset=['mensagem_limpa'])
print(f"Processamento concluído. {len(df_final):,} mensagens válidas retidas.")

## 3. Resultado Final: Mensagens Válidas

In [ ]:
pd.options.display.max_colwidth = 150
df_final[['autor', 'mensagem', 'mensagem_limpa']].head(50)